# 56. 전체 실험 비교와 최종 모델 선택

49-55에서 저장한 산출물을 모두 모아 비교합니다. 각 노트북은 공통적으로 `runs/<experiment_name>/config.json`, `metrics.json`, `pred_samples/`를 남기며, 56장은 이 파일들을 읽어 실험표, 성능 그래프, 예측 시각화를 한 번에 확인합니다.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "seg7_utils.py").exists():
    NOTEBOOK_DIR = Path("Vision 기초/7장")

sys.path.append(str(NOTEBOOK_DIR))
DATA_ROOT = NOTEBOOK_DIR / "data" / "mini_shapes_seg"
RUNS_ROOT = NOTEBOOK_DIR / "runs"

from seg7_utils import *
set_korean_font()
set_seed(7)

## 56-1. 모든 실험 결과 불러오기

In [ ]:
rows = collect_run_table(RUNS_ROOT)
rows

## 56-2. 실험 성능표 만들기

In [ ]:
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    metric_df = df.dropna(subset=["mean_iou"]).sort_values("mean_iou", ascending=False)
    display(metric_df)
except ImportError:
    pd = None
    metric_df = sorted([row for row in rows if row.get("mean_iou") is not None], key=lambda x: x["mean_iou"], reverse=True)
    metric_df

## 56-3. Metric bar chart

In [ ]:
import matplotlib.pyplot as plt

if pd is not None:
    plot_df = metric_df.copy()
    labels = plot_df["run"].tolist()
    miou = plot_df["mean_iou"].tolist()
    pacc = plot_df["pixel_accuracy"].tolist()
else:
    labels = [row["run"] for row in metric_df]
    miou = [row["mean_iou"] for row in metric_df]
    pacc = [row["pixel_accuracy"] for row in metric_df]

x = np.arange(len(labels))
plt.figure(figsize=(11, 4))
plt.bar(x - 0.18, miou, width=0.36, label="mean IoU")
plt.bar(x + 0.18, pacc, width=0.36, label="pixel accuracy")
plt.xticks(x, labels, rotation=30, ha="right")
plt.ylim(0, 1)
plt.ylabel("score")
plt.grid(True, axis="y", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 56-4. 예측 sample 나란히 비교

In [ ]:
from PIL import Image

candidate_runs = [
    "50_baseline_fcn",
    "53_loss_ce_dice",
    "54_aug_resolution",
    "55_tiny_unet_improved",
]

fig, axes = plt.subplots(len(candidate_runs), 1, figsize=(12, 3 * len(candidate_runs)))
if len(candidate_runs) == 1:
    axes = [axes]
for ax, run_name in zip(axes, candidate_runs):
    image_path = RUNS_ROOT / run_name / "pred_samples" / "sample_00.png"
    if image_path.exists():
        ax.imshow(Image.open(image_path))
        ax.set_title(run_name)
    else:
        ax.text(0.5, 0.5, f"missing: {run_name}", ha="center", va="center")
    ax.axis("off")
fig.tight_layout()
plt.show()

## 56-5. 최종 모델 선택

In [ ]:
valid_rows = [row for row in rows if row.get("mean_iou") is not None]
best = max(valid_rows, key=lambda row: row["mean_iou"])

print("최종 후보:", best["run"])
print("model:", best["model"])
print("loss:", best["loss"])
print("augment:", best["augment"])
print("mean IoU:", best["mean_iou"])
print("pixel accuracy:", best["pixel_accuracy"])

save_json(RUNS_ROOT / "56_final_comparison" / "config.json", {
    "experiment_name": "56_final_comparison",
    "source_runs": [row["run"] for row in rows],
})
save_json(RUNS_ROOT / "56_final_comparison" / "metrics.json", {
    "experiment_name": "56_final_comparison",
    "best_run": best,
    "all_runs": rows,
    "pixel_accuracy": best.get("pixel_accuracy"),
    "mean_iou": best.get("mean_iou"),
})